In [3]:
# ============================================================
# 16_report_assets_tables_figures.ipynb
# Report-ready figures and tables for TFG memory
#
# - RespiCast comparison uses MAE, scaled AE and RMSE.
# - Zero AE cases are audited, not removed.
# - Extreme / non-finite predictions are audited separately.
# ============================================================

from pathlib import Path
from datetime import datetime
import warnings
import math
import re
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# ============================================================
# 0. CONFIGURATION
# ============================================================

COMMON6 = ["BE", "CZ", "EE", "LT", "RO", "SI"]

PRIMARY_MODEL_METRIC = "mase"
FALLBACK_MODEL_METRIC = "mae"

FIG_DPI = 300
SAVE_PDF = True
SAVE_PNG = True

ENGLISH_LABELS = True
FORECAST_EXAMPLE_H = 1

EDA_START = None
EDA_END = None

RESULTS_SEARCH_ROOT = None

# RespiCast final clean comparison
REQUIRE_RESPICAST_CLEAN = True

# ============================================================
# 1. PROJECT ROOT AND OUTPUT FOLDERS
# ============================================================

def find_project_root():
    current = Path.cwd().resolve()
    candidates = [current] + list(current.parents)
    for c in candidates:
        if (c / "data").exists() and (c / "results").exists():
            return c
    raise FileNotFoundError(
        "Could not find project root. Run this notebook from the project folder "
        "or from the notebooks/ directory."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"

if RESULTS_SEARCH_ROOT is None:
    RESULTS_SEARCH_ROOT = RESULTS_DIR
else:
    RESULTS_SEARCH_ROOT = Path(RESULTS_SEARCH_ROOT)

ASSET_DIR = RESULTS_DIR / "report_assets"
FIG_DIR = ASSET_DIR / "figures"
TABLE_DIR = ASSET_DIR / "tables"
AUDIT_DIR = ASSET_DIR / "audits"

# Full overwrite of this notebook outputs
if ASSET_DIR.exists():
    shutil.rmtree(ASSET_DIR)

for d in [ASSET_DIR, FIG_DIR, TABLE_DIR, AUDIT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("RESULTS_DIR:", RESULTS_DIR)
print("OUTPUT:", ASSET_DIR)

# ============================================================
# 2. PLOTTING STYLE
# ============================================================

plt.rcParams.update({
    "figure.figsize": (10, 5.5),
    "figure.dpi": 120,
    "savefig.dpi": FIG_DPI,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "lines.linewidth": 1.7,
})

DISEASE_COLORS = {
    "ARI": "#1f77b4",
    "ILI": "#d62728",
}

SOURCE_COLORS = {
    "TFG": "#1f77b4",
    "RespiCast": "#ff7f0e",
}

# ============================================================
# 3. HELPER FUNCTIONS
# ============================================================

generated_assets = []

def save_figure(fig, name, description="", section_hint=""):
    if SAVE_PNG:
        png_path = FIG_DIR / f"{name}.png"
        fig.savefig(png_path, bbox_inches="tight")
        generated_assets.append({
            "type": "figure",
            "name": name,
            "path": str(png_path.relative_to(PROJECT_ROOT)),
            "description": description,
            "section_hint": section_hint,
        })
    if SAVE_PDF:
        pdf_path = FIG_DIR / f"{name}.pdf"
        fig.savefig(pdf_path, bbox_inches="tight")
        generated_assets.append({
            "type": "figure",
            "name": name,
            "path": str(pdf_path.relative_to(PROJECT_ROOT)),
            "description": description,
            "section_hint": section_hint,
        })
    plt.close(fig)

def save_table(df, name, description="", section_hint="", index=False, latex=True, max_latex_rows=60):
    csv_path = TABLE_DIR / f"{name}.csv"
    df.to_csv(csv_path, index=index)
    generated_assets.append({
        "type": "table",
        "name": name,
        "path": str(csv_path.relative_to(PROJECT_ROOT)),
        "description": description,
        "section_hint": section_hint,
    })
    if latex:
        tex_path = TABLE_DIR / f"{name}.tex"
        df_latex = df.copy()
        if len(df_latex) > max_latex_rows:
            df_latex = df_latex.head(max_latex_rows)
        with open(tex_path, "w", encoding="utf-8") as f:
            f.write(
                df_latex.to_latex(
                    index=index,
                    escape=True,
                    float_format=lambda x: f"{x:.3f}" if pd.notna(x) else "",
                )
            )
        generated_assets.append({
            "type": "latex_table",
            "name": name,
            "path": str(tex_path.relative_to(PROJECT_ROOT)),
            "description": description,
            "section_hint": section_hint,
        })

def read_csv_robust(path, nrows=None):
    try:
        return pd.read_csv(path, nrows=nrows)
    except UnicodeDecodeError:
        return pd.read_csv(path, nrows=nrows, encoding="latin1")

def clean_model_name(x, max_len=45):
    s = str(x)
    replacements = {
        "DL_GlobalGRU_all_infections_all_countries": "Global GRU",
        "DL_GlobalTransformer_all_infections_all_countries": "Global Transformer",
        "DL_DiseaseTransformer_all_countries": "Disease Transformer",
        "DL_LocalTCN_each_country": "Local TCN",
        "rf_global_all_infections_all_countries": "Global RF",
        "RandomForest(lags=4)": "RF lags=4",
        "SARIMA(1,0,0)x(1,0,0)[52] (initial)": "SARIMA initial",
        "SARIMA(1,0,0)x(1,1,0)[52] (seasonal diff)": "SARIMA seasonal diff",
        "Airline SARIMA(0,1,1)x(0,1,1)[52]": "Airline SARIMA",
        "respicast-quantileBaseline": "RespiCast baseline",
        "respicast-hubEnsemble": "RespiCast hub ensemble",
        "ensemble_mean_5": "Ensemble mean",
        "ensemble_median_5": "Ensemble median",
    }
    for old, new in replacements.items():
        s = s.replace(old, new)
    if len(s) > max_len:
        s = s[:max_len-3] + "..."
    return s

def infer_disease_from_filename(path):
    name = path.name.lower()
    if "ari" in name:
        return "ARI"
    if "ili" in name:
        return "ILI"
    return None

def week_start_sunday(date):
    d = pd.Timestamp(date)
    return d - pd.Timedelta(days=(d.weekday() + 1) % 7)

def relpath(path):
    path = Path(path)
    try:
        return str(path.relative_to(PROJECT_ROOT))
    except Exception:
        return str(path)

# ============================================================
# 4. LOAD INCIDENCE DATA
# ============================================================

def load_incidence_data():
    latest_files = []
    for disease in ["ARI", "ILI"]:
        candidates = list(DATA_DIR.glob(f"latest-{disease}_incidence.csv"))
        candidates += list(DATA_DIR.glob(f"latest-{disease.lower()}_incidence.csv"))
        if candidates:
            latest_files.append(candidates[0])

    if len(latest_files) >= 2:
        files = latest_files
        source_mode = "latest_files"
    else:
        files = sorted(DATA_DIR.glob("*ARI_incidence.csv")) + sorted(DATA_DIR.glob("*ILI_incidence.csv"))
        files = [f for f in files if not f.name.lower().startswith("latest-")]
        source_mode = "dated_files"

    if not files:
        raise FileNotFoundError("No incidence files found in data/.")

    all_parts = []
    inventory = []

    for f in files:
        df = read_csv_robust(f)
        df.columns = [str(c).strip() for c in df.columns]
        cols_lower = {c.lower(): c for c in df.columns}

        disease = infer_disease_from_filename(f)
        if "disease" in cols_lower:
            disease_col = cols_lower["disease"]
            df["disease"] = df[disease_col].astype(str).str.upper()
        else:
            df["disease"] = disease

        date_candidates = ["truth_date", "date", "target_end_date", "target", "week", "time_value"]
        date_col = next((cols_lower[c] for c in date_candidates if c in cols_lower), None)
        if date_col is None:
            raise ValueError(f"No date column found in {f.name}. Columns: {df.columns.tolist()}")

        loc_candidates = ["location", "country", "geo_value"]
        loc_col = next((cols_lower[c] for c in loc_candidates if c in cols_lower), None)
        if loc_col is None:
            raise ValueError(f"No location column found in {f.name}. Columns: {df.columns.tolist()}")

        value_candidates = ["value", "incidence", "y", "truth", "observed"]
        value_col = next((cols_lower[c] for c in value_candidates if c in cols_lower), None)
        if value_col is None:
            numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
            if len(numeric_cols) == 1:
                value_col = numeric_cols[0]
            else:
                raise ValueError(f"No value column found in {f.name}. Columns: {df.columns.tolist()}")

        part = pd.DataFrame({
            "truth_date": pd.to_datetime(df[date_col], errors="coerce"),
            "location": df[loc_col].astype(str),
            "value": pd.to_numeric(df[value_col], errors="coerce"),
            "disease": df["disease"].astype(str).str.upper(),
            "source_file": f.name,
        })

        part = part.dropna(subset=["truth_date", "location", "value", "disease"])
        all_parts.append(part)

        inventory.append({
            "file": f.name,
            "rows_loaded": len(df),
            "rows_valid": len(part),
            "disease_inferred": disease,
            "date_col": date_col,
            "location_col": loc_col,
            "value_col": value_col,
        })

    truth = pd.concat(all_parts, ignore_index=True)

    truth = (
        truth.sort_values(["disease", "location", "truth_date", "source_file"])
        .drop_duplicates(["disease", "location", "truth_date"], keep="last")
        .reset_index(drop=True)
    )

    if EDA_START is not None:
        truth = truth[truth["truth_date"] >= pd.Timestamp(EDA_START)]
    if EDA_END is not None:
        truth = truth[truth["truth_date"] <= pd.Timestamp(EDA_END)]

    inv = pd.DataFrame(inventory)
    save_table(
        inv,
        "00_incidence_file_inventory",
        "Input incidence files used for report EDA.",
        "Data Collection",
        latex=False
    )

    print(f"Incidence data loaded using mode: {source_mode}")
    print("truth shape:", truth.shape)
    print("date range:", truth["truth_date"].min(), "to", truth["truth_date"].max())
    print("diseases:", sorted(truth["disease"].unique()))
    print("locations:", sorted(truth["location"].unique()))

    return truth

truth = load_incidence_data()

# ============================================================
# 5. DATA COVERAGE TABLES
# ============================================================

coverage_rows = []

for disease, dfd in truth.groupby("disease"):
    full_start = week_start_sunday(dfd["truth_date"].min())
    full_end = week_start_sunday(dfd["truth_date"].max())
    full_calendar = pd.date_range(full_start, full_end, freq="W-SUN")

    for loc, dfl in dfd.groupby("location"):
        observed = dfl["truth_date"].drop_duplicates()
        n_obs = observed.nunique()
        n_expected = len(full_calendar)
        coverage_rows.append({
            "disease": disease,
            "location": loc,
            "first_date": observed.min(),
            "last_date": observed.max(),
            "n_observed_weeks": n_obs,
            "n_expected_weeks_global_calendar": n_expected,
            "coverage_rate": n_obs / n_expected if n_expected else np.nan,
            "missing_weeks_global_calendar": n_expected - n_obs,
            "mean_incidence": dfl["value"].mean(),
            "median_incidence": dfl["value"].median(),
            "std_incidence": dfl["value"].std(),
            "max_incidence": dfl["value"].max(),
        })

coverage = pd.DataFrame(coverage_rows).sort_values(["disease", "location"])

save_table(
    coverage,
    "01_data_coverage_by_country_disease",
    "Coverage and descriptive statistics by country and disease.",
    "Data Collection / Data Analysis",
)

disease_coverage = (
    coverage.groupby("disease", as_index=False)
    .agg(
        n_countries=("location", "nunique"),
        first_date=("first_date", "min"),
        last_date=("last_date", "max"),
        total_observed_weeks=("n_observed_weeks", "sum"),
        mean_country_coverage=("coverage_rate", "mean"),
        mean_incidence=("mean_incidence", "mean"),
        median_incidence=("median_incidence", "mean"),
    )
)

save_table(
    disease_coverage,
    "02_data_coverage_by_disease",
    "Overall data coverage by disease.",
    "Data Collection / Data Analysis",
)

display(disease_coverage)

# ============================================================
# 6. EDA FIGURES
# ============================================================

# Final modelling subsets used throughout the thesis
SELECTED_COUNTRIES_BY_DISEASE = {
    "ARI": ["BE", "BG", "CZ", "DE", "EE", "LT", "RO", "SI"],
    "ILI": ["BE", "CZ", "EE", "GR", "IE", "LT", "NL", "PL", "RO", "SI"],
}

def filter_selected_countries(df):
    parts = []
    for disease, countries in SELECTED_COUNTRIES_BY_DISEASE.items():
        dfd = df[(df["disease"] == disease) & (df["location"].isin(countries))].copy()
        parts.append(dfd)
    return pd.concat(parts, ignore_index=True)

truth_eda = filter_selected_countries(truth)

print("EDA figures restricted to final modelling subsets:")
for disease, countries in SELECTED_COUNTRIES_BY_DISEASE.items():
    dfd = truth_eda[truth_eda["disease"] == disease]
    print(
        disease,
        "countries:", sorted(dfd["location"].unique()),
        "| expected:", countries,
        "| rows:", len(dfd),
        "| max value:", dfd["value"].max()
    )

# Audit extreme values before plotting
# This does not remove observations automatically. It only documents the largest
# values so that potential data-entry errors can be checked explicitly.
top_values_all = (
    truth.sort_values("value", ascending=False)
    .head(50)
    .loc[:, ["disease", "location", "truth_date", "value", "source_file"]]
)

top_values_selected = (
    truth_eda.sort_values("value", ascending=False)
    .head(50)
    .loc[:, ["disease", "location", "truth_date", "value", "source_file"]]
)

save_table(
    top_values_all,
    "eda_audit_top50_values_all_countries",
    "Top 50 incidence values before restricting to the final modelling subsets.",
    "Data Analysis",
    latex=False,
)

save_table(
    top_values_selected,
    "eda_audit_top50_values_selected_countries",
    "Top 50 incidence values after restricting to the final modelling subsets.",
    "Data Analysis",
    latex=False,
)

display(top_values_selected.head(20))


# Fig 1: Average incidence rate over time, selected countries only
fig, ax = plt.subplots(figsize=(11, 5.5))

for disease in ["ARI", "ILI"]:
    dfd = truth_eda[truth_eda["disease"] == disease].copy()
    if dfd.empty:
        continue

    avg = (
        dfd.groupby("truth_date", as_index=False)["value"]
        .mean()
        .sort_values("truth_date")
    )

    ax.plot(
        avg["truth_date"],
        avg["value"],
        label=f"{disease} average incidence rate",
        color=DISEASE_COLORS.get(disease, None),
    )

ax.set_title("Average weekly incidence rate across selected countries")
ax.set_xlabel("Date")
ax.set_ylabel("Incidence rate")
ax.legend()
fig.autofmt_xdate()

save_figure(
    fig,
    "fig01_average_weekly_incidence",
    "Average weekly ARI and ILI incidence rate across selected countries.",
    "Data Analysis",
)


# Fig 2: Mean incidence by selected country and disease
for disease in ["ARI", "ILI"]:
    dfd = truth_eda[truth_eda["disease"] == disease].copy()
    if dfd.empty:
        continue

    mean_by_loc = (
        dfd.groupby("location", as_index=False)["value"]
        .mean()
        .sort_values("value", ascending=True)
    )

    fig, ax = plt.subplots(figsize=(8, max(4, 0.32 * len(mean_by_loc))))
    ax.barh(
        mean_by_loc["location"],
        mean_by_loc["value"],
        color=DISEASE_COLORS.get(disease, "#1f77b4"),
    )
    ax.set_title(f"Average weekly {disease} incidence rate by selected country")
    ax.set_xlabel("Mean incidence rate")
    ax.set_ylabel("Country")

    save_figure(
        fig,
        f"fig02_mean_incidence_by_country_{disease.lower()}",
        f"Average weekly {disease} incidence rate by selected country.",
        "Data Analysis",
    )


# Fig 3: Seasonal profile, selected countries only
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharey=False)

for ax, disease in zip(axes, ["ARI", "ILI"]):
    dfd = truth_eda[truth_eda["disease"] == disease].copy()
    if dfd.empty:
        ax.set_visible(False)
        continue

    iso = dfd["truth_date"].dt.isocalendar()
    dfd["iso_week"] = iso.week.astype(int)
    dfd = dfd[dfd["iso_week"].between(1, 52)]

    country_weekly = (
        dfd.groupby(["location", "iso_week"], as_index=False)["value"]
        .mean()
        .rename(columns={"value": "country_iso_week_mean"})
    )

    weekly = (
        country_weekly.groupby("iso_week")["country_iso_week_mean"]
        .agg(
            median="median",
            q10=lambda s: s.quantile(0.10),
            q90=lambda s: s.quantile(0.90),
            mean="mean",
            n_countries="count",
        )
        .reset_index()
    )

    ax.plot(
        weekly["iso_week"],
        weekly["median"],
        color=DISEASE_COLORS.get(disease),
        label="Median across selected countries",
    )

    ax.fill_between(
        weekly["iso_week"],
        weekly["q10"],
        weekly["q90"],
        color=DISEASE_COLORS.get(disease),
        alpha=0.18,
        label="10th-90th percentile across selected countries",
    )

    ax.set_title(f"{disease}: seasonal profile")
    ax.set_xlabel("ISO week")
    ax.set_ylabel("Incidence rate")
    ax.legend()

fig.suptitle("Seasonal incidence profile by disease across selected countries", y=1.02)
fig.tight_layout()

save_figure(
    fig,
    "fig03_seasonal_profile_by_disease",
    "Median seasonal profile and 10th-90th percentile band across selected countries by ISO week.",
    "Data Analysis / Time Series Analysis",
)


# Fig 4: Country correlation heatmaps, selected countries only
def plot_corr_heatmap(corr, title, name):
    labels = corr.columns.tolist()
    fig, ax = plt.subplots(figsize=(max(7, 0.45 * len(labels)), max(6, 0.45 * len(labels))))
    im = ax.imshow(corr.values, vmin=-1, vmax=1, cmap="coolwarm")
    ax.set_title(title)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=90)
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels)

    cbar = fig.colorbar(im, ax=ax, shrink=0.82)
    cbar.set_label("Pearson correlation")

    if len(labels) <= 16:
        for i in range(len(labels)):
            for j in range(len(labels)):
                val = corr.values[i, j]
                if pd.notna(val):
                    ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=7)

    fig.tight_layout()
    save_figure(fig, name, title, "Data Analysis / Time Series Analysis")


for disease in ["ARI", "ILI"]:
    dfd = truth_eda[truth_eda["disease"] == disease].copy()
    if dfd.empty:
        continue

    pivot = (
        dfd.pivot_table(index="truth_date", columns="location", values="value", aggfunc="mean")
        .sort_index()
    )

    # Keep the final country order defined above
    selected_order = [c for c in SELECTED_COUNTRIES_BY_DISEASE[disease] if c in pivot.columns]
    pivot = pivot[selected_order]

    corr = pivot.corr()

    save_table(
        corr.reset_index().rename(columns={"index": "location"}),
        f"03_country_correlation_matrix_{disease.lower()}",
        f"Correlation matrix of country-level {disease} incidence series for selected countries.",
        "Data Analysis",
        latex=False,
    )

    plot_corr_heatmap(
        corr,
        f"{disease}: correlation between selected country incidence series",
        f"fig04_country_correlation_heatmap_{disease.lower()}",
    )

# Fig 5: Missingness heatmap by year
missing_rows = []

for disease, dfd in truth.groupby("disease"):
    years = sorted(dfd["truth_date"].dt.year.unique())
    for loc, dfl in dfd.groupby("location"):
        observed_dates = set(pd.to_datetime(dfl["truth_date"]).dt.normalize())
        for year in years:
            start = pd.Timestamp(f"{year}-01-01")
            end = pd.Timestamp(f"{year}-12-31")
            cal = pd.date_range(week_start_sunday(start), week_start_sunday(end), freq="W-SUN")
            if len(cal) == 0:
                continue
            n_obs = sum(pd.Timestamp(x).normalize() in observed_dates for x in cal)
            missing_rows.append({
                "disease": disease,
                "location": loc,
                "year": year,
                "expected_weeks": len(cal),
                "observed_weeks": n_obs,
                "missing_rate": 1 - n_obs / len(cal),
            })

missing_by_year = pd.DataFrame(missing_rows)

save_table(
    missing_by_year,
    "04_missingness_by_country_year",
    "Missingness by country, year and disease.",
    "Data Collection / Data Analysis",
    latex=False,
)

for disease in sorted(missing_by_year["disease"].unique()):
    dfd = missing_by_year[missing_by_year["disease"] == disease]
    mat = dfd.pivot_table(index="location", columns="year", values="missing_rate", aggfunc="mean")
    mat = mat.sort_index()

    fig, ax = plt.subplots(figsize=(9, max(4, 0.35 * len(mat))))
    im = ax.imshow(mat.values, vmin=0, vmax=1, cmap="Reds")
    ax.set_title(f"{disease}: missingness rate by country and year")
    ax.set_xticks(range(len(mat.columns)))
    ax.set_xticklabels(mat.columns, rotation=45)
    ax.set_yticks(range(len(mat.index)))
    ax.set_yticklabels(mat.index)
    cbar = fig.colorbar(im, ax=ax, shrink=0.85)
    cbar.set_label("Missing rate")

    fig.tight_layout()

    save_figure(
        fig,
        f"fig05_missingness_heatmap_{disease.lower()}",
        f"Missingness heatmap for {disease}.",
        "Data Collection / Data Analysis",
    )

# Fig 6: ACF for representative countries
def autocorr_values(series, max_lag=52):
    x = pd.Series(series).astype(float)
    return np.array([x.autocorr(lag=lag) for lag in range(1, max_lag + 1)])

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)

for ax, disease in zip(axes, ["ARI", "ILI"]):
    dfd = truth[truth["disease"] == disease].copy()
    if dfd.empty:
        ax.set_visible(False)
        continue

    tmp = (
        dfd.groupby("location")
        .agg(n=("value", "size"), mean_value=("value", "mean"))
        .sort_values(["n", "mean_value"], ascending=[False, False])
    )
    loc = tmp.index[0]

    s = (
        dfd[dfd["location"] == loc]
        .sort_values("truth_date")
        .drop_duplicates("truth_date")
        .set_index("truth_date")["value"]
        .asfreq("W-SUN")
        .interpolate(limit_direction="both")
    )

    acf = autocorr_values(s, max_lag=52)
    ax.bar(np.arange(1, 53), acf, color=DISEASE_COLORS.get(disease, "#1f77b4"))
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_title(f"{disease}: autocorrelation example ({loc})")
    ax.set_ylabel("ACF")
    ax.set_ylim(-1, 1)

axes[-1].set_xlabel("Lag in weeks")
fig.tight_layout()

save_figure(
    fig,
    "fig06_acf_representative_series",
    "Autocorrelation functions for representative ARI and ILI country series.",
    "Data Analysis / Time Series Analysis",
)

# ============================================================
# 7. LOAD FINAL MODEL RESULT TABLES 
# ============================================================

FINAL_TEST_DIR = RESULTS_DIR / "final_test_2024_2025"

INTERNAL_GLOBAL_FILE = FINAL_TEST_DIR / "finaltest_horizon_summary.csv"
INTERNAL_COMMON6_FILE = FINAL_TEST_DIR / "finaltest_common6_horizon_summary.csv"

required_internal_files = [INTERNAL_GLOBAL_FILE, INTERNAL_COMMON6_FILE]
missing_internal = [p for p in required_internal_files if not p.exists()]

if missing_internal:
    raise FileNotFoundError(
        "Missing final internal result files:\n" +
        "\n".join(str(p) for p in missing_internal)
    )

print("Internal result files used:")
print("-", INTERNAL_GLOBAL_FILE)
print("-", INTERNAL_COMMON6_FILE)


def read_internal_horizon_summary(path: Path, scope: str) -> pd.DataFrame:
    df = read_csv_robust(path)
    df.columns = [str(c).strip() for c in df.columns]

    rename = {}
    for c in df.columns:
        cl = c.lower()
        if cl == "mae":
            rename[c] = "mae"
        elif cl == "rmse":
            rename[c] = "rmse"
        elif cl == "mase":
            rename[c] = "mase"
        elif cl == "disease":
            rename[c] = "disease"
        elif cl == "model":
            rename[c] = "model"
        elif cl == "h":
            rename[c] = "h"
        elif cl == "n_countries":
            rename[c] = "n_countries"
        elif cl == "n_points":
            rename[c] = "n_points"

    df = df.rename(columns=rename)
    df = df.loc[:, ~df.columns.duplicated()].copy()

    required = {"disease", "h", "model", "mae", "rmse", "mase", "n_countries", "n_points"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{path.name} is missing required columns: {sorted(missing)}")

    df["scope"] = scope
    df["disease"] = df["disease"].astype(str).str.upper()
    df["model"] = df["model"].astype(str)
    df["h"] = pd.to_numeric(df["h"], errors="raise").astype(int)

    for c in ["mae", "rmse", "mase", "n_countries", "n_points"]:
        df[c] = pd.to_numeric(df[c], errors="raise")

    df["model_short"] = df["model"].apply(clean_model_name)
    df["source_file"] = relpath(path)
    df["source_level"] = "final_validated_horizon_summary"

    out_cols = [
        "scope", "disease", "h", "model", "model_short",
        "mae", "rmse", "mase", "n_countries", "n_points",
        "source_level", "source_file"
    ]

    return df[out_cols].sort_values(["scope", "disease", "h", "mase", "mae"]).reset_index(drop=True)


global_results = read_internal_horizon_summary(INTERNAL_GLOBAL_FILE, "global")
common6_results = read_internal_horizon_summary(INTERNAL_COMMON6_FILE, "common6")

all_model_results = pd.concat([global_results, common6_results], ignore_index=True)
metric_for_sorting = PRIMARY_MODEL_METRIC if PRIMARY_MODEL_METRIC in all_model_results.columns else FALLBACK_MODEL_METRIC

model_summary_inventory = pd.DataFrame([
    {
        "file": relpath(INTERNAL_GLOBAL_FILE),
        "scope": "global",
        "rows": len(global_results),
        "models": global_results["model"].nunique(),
        "diseases": ",".join(sorted(global_results["disease"].unique())),
        "horizons": ",".join(map(str, sorted(global_results["h"].unique()))),
    },
    {
        "file": relpath(INTERNAL_COMMON6_FILE),
        "scope": "common6",
        "rows": len(common6_results),
        "models": common6_results["model"].nunique(),
        "diseases": ",".join(sorted(common6_results["disease"].unique())),
        "horizons": ",".join(map(str, sorted(common6_results["h"].unique()))),
    },
])

save_table(
    model_summary_inventory,
    "05_model_summary_file_inventory",
    "Inventory of the exact final model summary files used for report result tables.",
    "Results",
    latex=False,
)

save_table(
    all_model_results,
    "06_all_internal_model_results_global_common6",
    "Internal TFG model results loaded strictly from the final global and common6 horizon summaries.",
    "Results",
    latex=False,
)

top5_internal = (
    all_model_results
    .sort_values(["scope", "disease", "h", metric_for_sorting, "mae"])
    .groupby(["scope", "disease", "h"], group_keys=False)
    .head(5)
    .reset_index(drop=True)
)

save_table(
    top5_internal,
    "07_internal_top5_models_by_scope_disease_horizon",
    f"Top 5 internal models by {metric_for_sorting.upper()} for each scope, disease and horizon.",
    "Results",
)

winners_internal = (
    all_model_results
    .sort_values(["scope", "disease", "h", metric_for_sorting, "mae"])
    .groupby(["scope", "disease", "h"], as_index=False)
    .first()
)

save_table(
    winners_internal,
    "08_internal_winners_by_scope_disease_horizon",
    f"Winning internal model by {metric_for_sorting.upper()} for each scope, disease and horizon.",
    "Results",
)

print("\nInternal global result coverage:")
display(global_results.groupby(["disease", "h"]).agg(
    n_models=("model", "nunique"),
    n_countries_min=("n_countries", "min"),
    n_countries_max=("n_countries", "max"),
    n_points_min=("n_points", "min"),
    n_points_max=("n_points", "max"),
).reset_index())

print("\nInternal common6 result coverage:")
display(common6_results.groupby(["disease", "h"]).agg(
    n_models=("model", "nunique"),
    n_countries_min=("n_countries", "min"),
    n_countries_max=("n_countries", "max"),
    n_points_min=("n_points", "min"),
    n_points_max=("n_points", "max"),
).reset_index())

print("\nInternal winners:")
display(winners_internal)

# ============================================================
# 8. INTERNAL RESULT FIGURES
# ============================================================

def plot_internal_metric_heatmap(df: pd.DataFrame, scope: str, disease: str, metric: str):
    dfd = df[(df["scope"] == scope) & (df["disease"] == disease)].copy()

    if dfd.empty:
        print(f"Skipping heatmap: no data for {scope}, {disease}.")
        return

    # Order models by average value of the selected metric across available horizons.
    model_order = (
        dfd.groupby("model_short", as_index=False)
        .agg(mean_metric=(metric, "mean"))
        .sort_values(["mean_metric", "model_short"])["model_short"]
        .tolist()
    )

    pivot = (
        dfd.pivot_table(index="model_short", columns="h", values=metric, aggfunc="mean")
        .reindex(model_order)
        .sort_index(axis=1)
    )

    missing_cells = int(pivot.isna().sum().sum())
    if missing_cells > 0:
        print(
            f"WARNING: {scope} {disease} {metric} heatmap has {missing_cells} missing cells. "
            "Those missing cells come directly from the final horizon summary file."
        )

    fig, ax = plt.subplots(figsize=(8, max(4.2, 0.38 * len(pivot))))
    im = ax.imshow(pivot.values, cmap="viridis_r", aspect="auto")

    ax.set_title(f"{disease} - {scope}: {metric.upper()} by model and horizon")
    ax.set_xlabel("Forecast horizon")
    ax.set_ylabel("Model")
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([f"h={int(c)}" for c in pivot.columns])
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)

    finite_values = pivot.values[np.isfinite(pivot.values)]
    threshold = np.nanmean(finite_values) if len(finite_values) else 0

    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            val = pivot.values[i, j]
            if pd.notna(val):
                ax.text(
                    j, i, f"{val:.2f}",
                    ha="center",
                    va="center",
                    fontsize=7,
                    color="white" if val > threshold else "black",
                )

    cbar = fig.colorbar(im, ax=ax, shrink=0.82)
    cbar.set_label(metric.upper())
    fig.tight_layout()

    save_figure(
        fig,
        f"fig07_{metric}_heatmap_{scope}_{disease.lower()}",
        f"{metric.upper()} heatmap by model and horizon for {disease}, {scope}. "
        "Generated strictly from the final horizon summary CSV.",
        "Results",
    )

    save_table(
        pivot.reset_index(),
        f"24_plot_data_fig07_{metric}_heatmap_{scope}_{disease.lower()}",
        f"Exact data used for fig07 {metric} heatmap for {scope}, {disease}.",
        "Results / Plot Data",
        latex=False,
    )


for scope in ["global", "common6"]:
    for disease in ["ARI", "ILI"]:
        plot_internal_metric_heatmap(all_model_results, scope, disease, metric_for_sorting)

# ============================================================
# 9. INTERNAL RESULT SOURCE AUDIT
# ============================================================

internal_source_audit = pd.DataFrame([
    {
        "output": "internal result tables and fig07 heatmaps",
        "scope": "global",
        "source_file": relpath(INTERNAL_GLOBAL_FILE),
        "note": "Complete global final-test horizon summary. No top-k or partial table is used.",
    },
    {
        "output": "internal result tables and fig07 heatmaps",
        "scope": "common6",
        "source_file": relpath(INTERNAL_COMMON6_FILE),
        "note": "Complete common6 final-test horizon summary. No top-k or partial table is used.",
    },
])

save_table(
    internal_source_audit,
    "25_internal_result_figure_source_audit",
    "Audit documenting the exact source files used for internal result figures.",
    "Results / Audit",
    latex=False,
)


# ============================================================
# 10. FORECAST EXAMPLE PLOTS
# ============================================================

def find_forecast_prediction_file():
    candidates = []

    preferred_names = [
        "finaltest_all_predictions_long.csv",
        "04_combined_tfg_respicast_predictions_long.csv",
        "06_combined_tfg_respicast_predictions_long_corrected.csv",
    ]

    for name in preferred_names:
        candidates += list(RESULTS_DIR.rglob(name))

    if candidates:
        candidates = sorted(candidates, key=lambda p: p.stat().st_mtime, reverse=True)
        return candidates[0]

    patterns = [
        "*final_predictions*.csv",
        "*rolling_2024_2025*.csv",
        "*test_2024_2025*.csv",
    ]

    matches = []
    for pat in patterns:
        matches += list(RESULTS_DIR.rglob(pat))

    matches = [p for p in matches if p.suffix.lower() == ".csv"]

    if matches:
        return sorted(matches, key=lambda p: p.stat().st_mtime, reverse=True)[0]

    return None

prediction_file = find_forecast_prediction_file()

if prediction_file is not None:
    print("Forecast example prediction file:", prediction_file)

    pred_df = read_csv_robust(prediction_file)
    pred_df.columns = [c.lower().strip() for c in pred_df.columns]

    required_pred_cols = {"origin", "target", "disease", "location", "h", "y", "pred", "model"}

    if required_pred_cols.issubset(set(pred_df.columns)):
        if "source" in pred_df.columns:
            pred_df = pred_df[pred_df["source"].astype(str).str.upper() == "TFG"].copy()

        pred_df["origin"] = pd.to_datetime(pred_df["origin"], errors="coerce")
        pred_df["target"] = pd.to_datetime(pred_df["target"], errors="coerce")
        pred_df["h"] = pd.to_numeric(pred_df["h"], errors="coerce")
        pred_df["y"] = pd.to_numeric(pred_df["y"], errors="coerce")
        pred_df["pred"] = pd.to_numeric(pred_df["pred"], errors="coerce")
        pred_df["disease"] = pred_df["disease"].astype(str).str.upper()
        pred_df["location"] = pred_df["location"].astype(str)

        pred_df = pred_df.dropna(subset=["origin", "target", "h", "y", "pred", "disease", "location", "model"])
        pred_df["h"] = pred_df["h"].astype(int)
        pred_df = pred_df[np.isfinite(pred_df["pred"])].copy()
        pred_df["AE"] = (pred_df["y"] - pred_df["pred"]).abs()

        forecast_inventory = (
            pred_df.groupby(["disease", "h", "model"], as_index=False)
            .agg(
                n=("AE", "size"),
                mean_AE=("AE", "mean"),
                first_origin=("origin", "min"),
                last_origin=("origin", "max"),
                first_target=("target", "min"),
                last_target=("target", "max"),
                n_countries=("location", "nunique"),
            )
            .sort_values(["disease", "h", "mean_AE"])
        )

        save_table(
            forecast_inventory,
            "09_forecast_prediction_inventory",
            "Forecast prediction inventory used for visual examples.",
            "Results",
            latex=False,
        )

        examples = []

        for disease in ["ARI", "ILI"]:
            dfd = pred_df[(pred_df["disease"] == disease) & (pred_df["h"] == FORECAST_EXAMPLE_H)].copy()
            if dfd.empty:
                continue

            best_model = (
                dfd.groupby("model")["AE"]
                .mean()
                .sort_values()
                .index[0]
            )

            loc_rank = (
                dfd[dfd["model"] == best_model]
                .groupby("location")
                .agg(n=("y", "size"), mean_y=("y", "mean"))
                .sort_values(["n", "mean_y"], ascending=[False, False])
            )

            loc = loc_rank.index[0]

            g = (
                dfd[(dfd["model"] == best_model) & (dfd["location"] == loc)]
                .sort_values("target")
                .drop_duplicates(["target"], keep="last")
            )

            examples.append((disease, best_model, loc, g))

        if examples:
            fig, axes = plt.subplots(len(examples), 1, figsize=(12, 4.2 * len(examples)), sharex=False)

            if len(examples) == 1:
                axes = [axes]

            for ax, (disease, best_model, loc, g) in zip(axes, examples):
                ax.plot(g["target"], g["y"], label="Observed", color="black", linewidth=1.6)
                ax.plot(
                    g["target"],
                    g["pred"],
                    label=f"Forecast: {clean_model_name(best_model)}",
                    color=DISEASE_COLORS.get(disease),
                    alpha=0.9
                )
                ax.set_title(f"{disease} h={FORECAST_EXAMPLE_H}: observed vs forecast in {loc}")
                ax.set_xlabel("Target date")
                ax.set_ylabel("Incidence")
                ax.legend()
                ax.grid(alpha=0.25)

            fig.tight_layout()

            save_figure(
                fig,
                "fig09_forecast_examples_observed_vs_predicted",
                "Observed vs predicted incidence for representative countries and best h=1 models.",
                "Results",
            )

            fig, axes = plt.subplots(len(examples), 1, figsize=(12, 4.2 * len(examples)), sharex=False)

            if len(examples) == 1:
                axes = [axes]

            for ax, (disease, best_model, loc, g) in zip(axes, examples):
                gz = g.tail(20) if len(g) > 16 else g

                ax.plot(gz["target"], gz["y"], marker="o", label="Observed", color="black", linewidth=1.6)
                ax.plot(
                    gz["target"],
                    gz["pred"],
                    marker="o",
                    label=f"Forecast: {clean_model_name(best_model)}",
                    color=DISEASE_COLORS.get(disease),
                    alpha=0.9
                )
                ax.set_title(f"{disease} h={FORECAST_EXAMPLE_H}: zoomed forecast example in {loc}")
                ax.set_xlabel("Target date")
                ax.set_ylabel("Incidence")
                ax.legend()
                ax.grid(alpha=0.25)

            fig.tight_layout()

            save_figure(
                fig,
                "fig10_forecast_examples_zoomed",
                "Zoomed observed vs predicted incidence examples.",
                "Results",
            )

    else:
        print("Prediction file found, but required columns are missing:", prediction_file)
else:
    print("No forecast prediction file found. Forecast example plots skipped.")

# ============================================================
# 11. FINAL RESPICAST COMPARISON TABLES AND FIGURES
# ============================================================
# This section uses the final clean RespiCast comparison folder only:
#   results/final_test_2024_2025/respicast_final_clean_tables/
#
# The main report figures are generated strictly from:
#   09_MAIN_top3_TFG_vs_top2_RespiCast_plus_baseline.csv

# This keeps the plots aligned with the final table 

RESPICAST_CLEAN_DIR = FINAL_TEST_DIR / "respicast_final_clean_tables"
RESPICAST_MAIN_FILE = RESPICAST_CLEAN_DIR / "09_MAIN_top3_TFG_vs_top2_RespiCast_plus_baseline.csv"

if not RESPICAST_MAIN_FILE.exists():
    msg = (
        "Missing final clean RespiCast comparison table:\n"
        f"{RESPICAST_MAIN_FILE}\n"
        "Run the final RespiCast comparison notebook first."
    )
    if REQUIRE_RESPICAST_CLEAN:
        raise FileNotFoundError(msg)
    else:
        print(msg)
else:
    print("Final RespiCast clean dir:", RESPICAST_CLEAN_DIR)
    print("Main RespiCast table used:", RESPICAST_MAIN_FILE)

    paths = {
        "column_dictionary": RESPICAST_CLEAN_DIR / "00_column_dictionary.csv",
        "horizon_audit": RESPICAST_CLEAN_DIR / "01_horizon_date_audit.csv",
        "prediction_quality": RESPICAST_CLEAN_DIR / "02_prediction_quality_audit.csv",
        "invalid_extreme_rows": RESPICAST_CLEAN_DIR / "03_invalid_or_extreme_prediction_rows.csv",
        "duplicate_audit": RESPICAST_CLEAN_DIR / "04_duplicate_prediction_key_audit.csv",
        "scales_used": RESPICAST_CLEAN_DIR / "05_scales_used.csv",
        "paired": RESPICAST_CLEAN_DIR / "06_paired_forecasts_vs_baseline.csv",
        "zero_audit": RESPICAST_CLEAN_DIR / "07_zero_error_audit.csv",
        "metric_summary": RESPICAST_CLEAN_DIR / "08_metric_summary_all_methods.csv",
        "main": RESPICAST_MAIN_FILE,
        "top10_all": RESPICAST_CLEAN_DIR / "10_top10_all_methods_by_scope_disease_horizon.csv",
        "top10_coverage": RESPICAST_CLEAN_DIR / "11_top10_coverage_filtered.csv",
        "tfg_positions": RESPICAST_CLEAN_DIR / "12_tfg_positions_only.csv",
        "respicast_positions": RESPICAST_CLEAN_DIR / "13_respicast_positions_only.csv",
        "reference_universe": RESPICAST_CLEAN_DIR / "14_reference_universe.csv",
    }

    loaded = {}
    for key, p in paths.items():
        if p.exists():
            loaded[key] = read_csv_robust(p)
            print(f"Loaded {key}: {loaded[key].shape} from {p.name}")
        else:
            loaded[key] = pd.DataFrame()
            print(f"Missing optional file: {p.name}")

    resp_main = loaded["main"].copy()
    resp_metric = loaded["metric_summary"].copy()
    resp_top10 = loaded["top10_coverage"].copy() if not loaded["top10_coverage"].empty else loaded["top10_all"].copy()
    resp_tfg = loaded["tfg_positions"].copy()
    resp_zero = loaded["zero_audit"].copy()
    resp_quality = loaded["prediction_quality"].copy()
    resp_extreme = loaded["invalid_extreme_rows"].copy()
    resp_dict = loaded["column_dictionary"].copy()
    resp_ref = loaded["reference_universe"].copy()
    resp_scales = loaded["scales_used"].copy()

    # Main table validation,keep original column names because these are the names used in the final CSV
    required_resp_main_cols = {
        "scope", "disease", "h", "main_table_rank", "source", "model",
        "mean_AE", "mean_scaled_AE", "RMSE", "median_AE",
        "n_pairs", "reference_pairs", "coverage_vs_baseline",
        "n_countries", "mean_AE_ref_baseline",
        "MAE_ratio_vs_baseline", "scaled_AE_ratio_vs_baseline",
        "n_negative_predictions", "first_origin", "last_origin",
        "first_target", "last_target",
    }

    missing_resp_cols = required_resp_main_cols - set(resp_main.columns)
    if missing_resp_cols:
        raise ValueError(
            "The final RespiCast main table is missing required columns: "
            f"{sorted(missing_resp_cols)}"
        )

    resp_main["scope"] = resp_main["scope"].astype(str)
    resp_main["disease"] = resp_main["disease"].astype(str).str.upper()
    resp_main["source"] = resp_main["source"].astype(str)
    resp_main["model"] = resp_main["model"].astype(str)
    resp_main["model_short"] = resp_main["model"].apply(clean_model_name)

    for c in [
        "h", "main_table_rank", "mean_AE", "mean_scaled_AE", "RMSE",
        "median_AE", "n_pairs", "reference_pairs", "coverage_vs_baseline",
        "n_countries", "mean_AE_ref_baseline", "MAE_ratio_vs_baseline",
        "scaled_AE_ratio_vs_baseline", "n_negative_predictions",
    ]:
        resp_main[c] = pd.to_numeric(resp_main[c], errors="raise")

    resp_main["h"] = resp_main["h"].astype(int)
    resp_main["main_table_rank"] = resp_main["main_table_rank"].astype(int)

    for c in ["first_origin", "last_origin", "first_target", "last_target"]:
        resp_main[c] = pd.to_datetime(resp_main[c], errors="raise")

    resp_main = resp_main.sort_values(["scope", "disease", "h", "main_table_rank"]).reset_index(drop=True)

    print("\nRespiCast main table coverage audit:")
    display(resp_main.groupby(["scope", "disease", "h"]).agg(
        n_rows=("model", "size"),
        n_tfg=("source", lambda x: (x == "TFG").sum()),
        n_respicast=("source", lambda x: (x == "RespiCast").sum()),
        n_pairs_min=("n_pairs", "min"),
        n_pairs_max=("n_pairs", "max"),
        reference_pairs_min=("reference_pairs", "min"),
        reference_pairs_max=("reference_pairs", "max"),
    ).reset_index())

    # Main report columns for external benchmark
    resp_main_cols = [
        "scope", "disease", "h", "main_table_rank",
        "source", "model", "mean_AE", "mean_scaled_AE", "RMSE", "median_AE",
        "n_pairs", "reference_pairs", "coverage_vs_baseline", "n_countries",
        "mean_AE_ref_baseline", "MAE_ratio_vs_baseline",
        "scaled_AE_ratio_vs_baseline", "n_negative_predictions",
        "first_origin", "last_origin", "first_target", "last_target",
    ]

    save_table(
        resp_main[resp_main_cols],
        "10_respicast_main_top3_tfg_top2_respicast_baseline",
        "Final simplified RespiCast comparison table using MAE and scaled AE.",
        "Results / External Benchmark",
        latex=False,
    )

    if not resp_metric.empty:
        metric_cols = [
            c for c in [
                "scope", "source", "model", "disease", "h",
                "mean_AE", "median_AE", "RMSE", "mean_scaled_AE", "median_scaled_AE",
                "mean_AE_ref_baseline", "mean_scaled_AE_ref_baseline",
                "MAE_ratio_vs_baseline", "scaled_AE_ratio_vs_baseline",
                "n_pairs", "reference_pairs", "coverage_vs_baseline",
                "n_countries", "rank_by_scaled_AE", "rank_by_AE",
                "first_origin", "last_origin",
            ] if c in resp_metric.columns
        ]

        save_table(
            resp_metric[metric_cols],
            "11_respicast_metric_summary_all_methods",
            "Full RespiCast comparison summary using MAE and scaled AE.",
            "Results / External Benchmark / Appendix",
            latex=False,
        )

    if not resp_top10.empty:
        top10_cols = [
            c for c in [
                "scope", "source", "model", "disease", "h",
                "mean_AE", "mean_scaled_AE", "RMSE",
                "n_pairs", "reference_pairs", "coverage_vs_baseline",
                "n_countries", "rank_by_scaled_AE", "rank_by_AE",
                "MAE_ratio_vs_baseline", "scaled_AE_ratio_vs_baseline",
            ] if c in resp_top10.columns
        ]

        save_table(
            resp_top10[top10_cols],
            "12_respicast_top10_coverage_filtered",
            "Top methods by scope, disease and horizon after coverage filtering.",
            "Results / External Benchmark",
            latex=False,
        )

    if not resp_tfg.empty:
        tfg_cols = [
            c for c in [
                "scope", "source", "model", "disease", "h",
                "mean_AE", "mean_scaled_AE", "RMSE",
                "n_pairs", "reference_pairs", "coverage_vs_baseline",
                "n_countries", "rank_by_scaled_AE", "rank_by_AE",
                "MAE_ratio_vs_baseline", "scaled_AE_ratio_vs_baseline",
            ] if c in resp_tfg.columns
        ]

        save_table(
            resp_tfg[tfg_cols],
            "13_respicast_tfg_positions_only",
            "Positions of TFG methods within the final RespiCast comparison.",
            "Results / External Benchmark",
            latex=False,
        )

    if not resp_zero.empty:
        save_table(
            resp_zero,
            "14_respicast_zero_error_audit",
            "Audit of zero absolute errors in the RespiCast comparison.",
            "Results / External Benchmark / Appendix",
            latex=False,
        )

    if not resp_quality.empty:
        save_table(
            resp_quality,
            "15_respicast_prediction_quality_audit",
            "Audit of non-finite, negative and extreme predictions in the RespiCast comparison.",
            "Results / External Benchmark / Appendix",
            latex=False,
        )

    if not resp_extreme.empty:
        save_table(
            resp_extreme,
            "16_respicast_invalid_or_extreme_prediction_rows",
            "Rows excluded or flagged due to invalid or extreme predictions.",
            "Results / External Benchmark / Appendix",
            latex=False,
        )

    if not resp_dict.empty:
        save_table(
            resp_dict,
            "17_respicast_column_dictionary",
            "Column dictionary for the final RespiCast comparison tables.",
            "Appendix",
            latex=False,
        )

    if not resp_ref.empty:
        save_table(
            resp_ref,
            "18_respicast_reference_universe",
            "Reference forecast universe used for pairing against respicast-quantileBaseline.",
            "Results / External Benchmark / Appendix",
            latex=False,
        )

    if not resp_scales.empty:
        save_table(
            resp_scales,
            "19_respicast_scales_used",
            "MASE/scaled-AE scales used in the external benchmark.",
            "Results / External Benchmark / Appendix",
            latex=False,
        )

    resp_source_audit = pd.DataFrame([
        {
            "output": "external benchmark tables and figures",
            "source_file": relpath(RESPICAST_MAIN_FILE),
            "note": "The main external benchmark figures are generated strictly from this final clean table.",
        }
    ])

    save_table(
        resp_source_audit,
        "26_respicast_result_figure_source_audit",
        "Audit documenting the exact source file used for RespiCast comparison figures.",
        "Results / External Benchmark / Audit",
        latex=False,
    )

# --------------------------------------------------------
# Fig 11: External benchmark ranking by scaled AE
# --------------------------------------------------------
for scope in sorted(resp_main["scope"].dropna().unique()):
    for disease in sorted(resp_main["disease"].dropna().unique()):
        dfd = resp_main[
            (resp_main["scope"] == scope) &
            (resp_main["disease"] == disease)
        ].copy()

        if dfd.empty:
            continue

        horizons = sorted(dfd["h"].dropna().unique())

        # Use a 2x2 layout for the four horizons to improve readability in the report.
        nrows, ncols = 2, 2
        fig, axes = plt.subplots(
            nrows,
            ncols,
            figsize=(13.5, 8.5),
            sharex=False,
            sharey=False,
        )

        axes_flat = axes.flatten()

        for ax, h in zip(axes_flat, horizons):
            g = (
                dfd[dfd["h"] == h]
                .sort_values("main_table_rank", ascending=True)
                .copy()
            )

            # Reverse order so that the best-ranked method appears at the top of the horizontal bar chart.
            g = g.iloc[::-1]
            colors = [SOURCE_COLORS.get(src, "#999999") for src in g["source"]]

            ax.barh(g["model_short"], g["mean_scaled_AE"], color=colors)
            ax.set_title(f"h={int(h)}")
            ax.set_xlabel("Mean scaled AE")
            ax.set_ylabel("")
            ax.grid(axis="x", alpha=0.25)

            max_x = g["mean_scaled_AE"].max()
            ax.set_xlim(0, max_x * 1.18)

            for y_pos, (_, row) in enumerate(g.iterrows()):
                ax.text(
                    row["mean_scaled_AE"],
                    y_pos,
                    f" {row['mean_scaled_AE']:.2f}",
                    va="center",
                    fontsize=8,
                )

        for ax in axes_flat[len(horizons):]:
            ax.set_visible(False)

        fig.suptitle(
            f"{scope} - {disease}: TFG vs RespiCast by mean scaled absolute error",
            y=0.98,
        )

        handles = [
            plt.Rectangle((0, 0), 1, 1, color=SOURCE_COLORS["TFG"], label="TFG"),
            plt.Rectangle((0, 0), 1, 1, color=SOURCE_COLORS["RespiCast"], label="RespiCast"),
        ]

        fig.legend(
            handles=handles,
            loc="lower center",
            ncol=2,
            bbox_to_anchor=(0.5, -0.01),
        )

        fig.tight_layout(rect=[0, 0.04, 1, 0.95])

        save_figure(
            fig,
            f"fig11_respicast_scaled_AE_ranking_{scope}_{disease.lower()}",
            f"Final external benchmark ranking by mean scaled AE for {scope}, {disease}. "
            "Generated strictly from 09_MAIN_top3_TFG_vs_top2_RespiCast_plus_baseline.csv.",
            "Results / External Benchmark",
        )

        save_table(
            dfd.sort_values(["h", "main_table_rank"]),
            f"27_plot_data_fig11_respicast_scaled_AE_ranking_{scope}_{disease.lower()}",
            f"Exact data used for fig11 RespiCast scaled AE ranking for {scope}, {disease}.",
            "Results / External Benchmark / Plot Data",
            latex=False,
        )

    # --------------------------------------------------------
    # Fig 12: Ratio vs RespiCast baseline
    # --------------------------------------------------------
    for scope in sorted(resp_main["scope"].dropna().unique()):
        for disease in sorted(resp_main["disease"].dropna().unique()):
            dfd = resp_main[
                (resp_main["scope"] == scope) &
                (resp_main["disease"] == disease)
            ].copy()

            if dfd.empty:
                continue

            horizons = sorted(dfd["h"].dropna().unique())
            fig, axes = plt.subplots(1, len(horizons), figsize=(4.8 * len(horizons), 5.2), sharex=False)

            if len(horizons) == 1:
                axes = [axes]

            for ax, h in zip(axes, horizons):
                g = (
                    dfd[dfd["h"] == h]
                    .sort_values("main_table_rank", ascending=True)
                    .copy()
                )

                g = g.iloc[::-1]
                colors = [SOURCE_COLORS.get(src, "#999999") for src in g["source"]]

                ax.barh(g["model_short"], g["scaled_AE_ratio_vs_baseline"], color=colors)
                ax.axvline(1.0, color="black", linewidth=0.9, linestyle="--")
                ax.set_title(f"h={int(h)}")
                ax.set_xlabel("Scaled AE ratio vs RespiCast baseline")
                ax.set_ylabel("")
                ax.grid(alpha=0.25)

                for y_pos, (_, row) in enumerate(g.iterrows()):
                    ax.text(
                        row["scaled_AE_ratio_vs_baseline"],
                        y_pos,
                        f" {row['scaled_AE_ratio_vs_baseline']:.2f}",
                        va="center",
                        fontsize=7,
                    )

            fig.suptitle(f"{scope} - {disease}: relative error vs RespiCast baseline", y=1.03)

            handles = [
                plt.Rectangle((0, 0), 1, 1, color=SOURCE_COLORS["TFG"], label="TFG"),
                plt.Rectangle((0, 0), 1, 1, color=SOURCE_COLORS["RespiCast"], label="RespiCast"),
            ]
            fig.legend(handles=handles, loc="lower center", ncol=2, bbox_to_anchor=(0.5, -0.06))

            fig.tight_layout()

            save_figure(
                fig,
                f"fig12_respicast_scaled_AE_ratio_{scope}_{disease.lower()}",
                f"Scaled AE ratio against respicast-quantileBaseline for {scope}, {disease}. "
                "Generated strictly from 09_MAIN_top3_TFG_vs_top2_RespiCast_plus_baseline.csv.",
                "Results / External Benchmark",
            )

            save_table(
                dfd.sort_values(["h", "main_table_rank"]),
                f"28_plot_data_fig12_respicast_scaled_AE_ratio_{scope}_{disease.lower()}",
                f"Exact data used for fig12 RespiCast scaled AE ratio for {scope}, {disease}.",
                "Results / External Benchmark / Plot Data",
                latex=False,
            )

    # --------------------------------------------------------
    # Fig 13: Coverage vs baseline
    # --------------------------------------------------------
    tmp = resp_main.copy()
    cov = (
        tmp.groupby(["source", "model", "model_short"], as_index=False)
        .agg(
            mean_coverage_vs_baseline=("coverage_vs_baseline", "mean"),
            min_coverage_vs_baseline=("coverage_vs_baseline", "min"),
            max_coverage_vs_baseline=("coverage_vs_baseline", "max"),
            mean_n_pairs=("n_pairs", "mean"),
        )
        .sort_values("mean_coverage_vs_baseline", ascending=True)
    )

    colors = [SOURCE_COLORS.get(src, "#999999") for src in cov["source"]]

    fig, ax = plt.subplots(figsize=(11, max(5.5, 0.35 * len(cov))))
    ax.barh(cov["model_short"], cov["mean_coverage_vs_baseline"], color=colors)
    ax.axvline(1.0, color="black", linewidth=0.9, linestyle="--")
    ax.set_title("External benchmark: average coverage relative to RespiCast baseline")
    ax.set_xlabel("Mean coverage vs baseline")
    ax.set_ylabel("Model")
    ax.grid(alpha=0.25)

    handles = [
        plt.Rectangle((0, 0), 1, 1, color=SOURCE_COLORS["TFG"], label="TFG"),
        plt.Rectangle((0, 0), 1, 1, color=SOURCE_COLORS["RespiCast"], label="RespiCast"),
    ]
    ax.legend(handles=handles, loc="lower right")

    fig.tight_layout()

    save_figure(
        fig,
        "fig13_respicast_coverage_vs_baseline",
        "Average coverage relative to respicast-quantileBaseline for methods in the simplified external table.",
        "Results / External Benchmark",
    )

    save_table(
        cov,
        "29_plot_data_fig13_respicast_coverage_vs_baseline",
        "Exact data used for fig13 RespiCast coverage plot.",
        "Results / External Benchmark / Plot Data",
        latex=False,
    )


# ============================================================
# 12. DIEBOLD-MARIANO TABLES
# ============================================================

def find_dm_files():
    patterns = [
        "*dm*winner*.csv",
        "*diebold*winner*.csv",
        "*dm_all_pairs*.csv",
        "*diebold*all_pairs*.csv",
    ]
    files = []
    for pat in patterns:
        files += list(RESULTS_DIR.rglob(pat))
    files = sorted(set(files), key=lambda p: p.stat().st_mtime, reverse=True)
    return files

dm_files = find_dm_files()
dm_report = pd.DataFrame()
dm_inventory_rows = []

if dm_files:
    for f in dm_files:
        try:
            d = read_csv_robust(f)
        except Exception:
            continue

        d.columns = [c.strip() for c in d.columns]

        dm_inventory_rows.append({
            "file": relpath(f),
            "rows": len(d),
            "columns": ", ".join(d.columns[:20]),
        })

        if any("winner" in c.lower() for c in d.columns) or "better_by_mean_loss" in [c.lower() for c in d.columns]:
            dm_report = d.copy()
            dm_source_file = f
            break

    dm_inventory = pd.DataFrame(dm_inventory_rows)

    save_table(
        dm_inventory,
        "20_dm_file_inventory",
        "DM-test files detected in results/.",
        "Results / Diebold-Mariano",
        latex=False,
    )

    if not dm_report.empty:
        possible_cols = [
            "scope", "loss", "disease", "h",
            "winner", "better_by_mean_loss",
            "model_A", "model_B",
            "competitor", "mean_loss_winner", "mean_loss_competitor",
            "mean_loss_A_panel", "mean_loss_B_panel",
            "mean_loss_diff_A_minus_B",
            "dm_stat", "p_one_sided",
            "p_one_sided_winner_better",
            "p_one_sided_winner_better_holm",
            "holm_reject_alpha_0_05",
            "significant_holm_0_05",
            "n_pair", "n",
        ]

        keep_cols = [c for c in possible_cols if c in dm_report.columns]
        dm_clean = dm_report[keep_cols].copy() if keep_cols else dm_report.copy()

        sort_cols = [c for c in ["scope", "disease", "h", "dm_stat"] if c in dm_clean.columns]

        if sort_cols:
            ascending = [True] * len(sort_cols)
            if "dm_stat" in sort_cols:
                ascending[-1] = False
            dm_clean = dm_clean.sort_values(sort_cols, ascending=ascending)

        save_table(
            dm_clean,
            "21_dm_test_simplified_report_table",
            "Simplified Diebold-Mariano test table.",
            "Results / Diebold-Mariano",
            latex=False,
        )

        sig_col = None
        for c in ["holm_reject_alpha_0_05", "significant_holm_0_05"]:
            if c in dm_clean.columns:
                sig_col = c
                break

        if sig_col is not None and all(c in dm_clean.columns for c in ["scope", "disease", "h"]):
            temp = dm_clean.copy()
            temp[sig_col] = temp[sig_col].astype(str).str.lower().isin(["true", "1", "yes", "y"])

            sig_summary = (
                temp.groupby(["scope", "disease", "h"], as_index=False)
                .agg(
                    n_tests=(sig_col, "size"),
                    n_significant=(sig_col, "sum"),
                )
            )
            sig_summary["share_significant"] = sig_summary["n_significant"] / sig_summary["n_tests"]

            save_table(
                sig_summary,
                "22_dm_significance_summary_by_scope_disease_horizon",
                "Summary of significant DM comparisons by scope, disease and horizon.",
                "Results / Diebold-Mariano",
            )

            fig, ax = plt.subplots(figsize=(9, 4.8))
            labels = sig_summary.apply(lambda r: f"{r['scope']} {r['disease']} h={int(r['h'])}", axis=1)
            ax.bar(labels, sig_summary["share_significant"])
            ax.set_ylim(0, 1)
            ax.set_ylabel("Share of significant comparisons")
            ax.set_title("DM-test significance after multiple-comparison correction")
            ax.tick_params(axis="x", rotation=70)
            fig.tight_layout()

            save_figure(
                fig,
                "fig14_dm_significance_summary",
                "Share of significant DM-test comparisons by scope, disease and horizon.",
                "Results / Diebold-Mariano",
            )

else:
    print("No DM-test files found. DM section skipped.")

# ============================================================
# 13. REPORT CATALOG
# ============================================================

catalog = pd.DataFrame(generated_assets)
catalog_path = ASSET_DIR / "report_assets_catalog.csv"
catalog.to_csv(catalog_path, index=False)

print("\n============================================================")
print("REPORT ASSETS GENERATED")
print("============================================================")
print("Output folder:", ASSET_DIR)
print("Catalog:", catalog_path)
print("\nFigures:", len(catalog[catalog["type"] == "figure"]))
print("Tables :", len(catalog[catalog["type"].str.contains("table", na=False)]))

display(catalog)

# ============================================================
# 14. SUGGESTED ORDER FOR THE MEMORY
# ============================================================

suggested_order = pd.DataFrame([
    {
        "thesis_section": "Data Collection / Data Preprocessing",
        "recommended_asset": "01_data_coverage_by_country_disease",
        "reason": "Documents which countries and diseases are available and the quality of the weekly panel."
    },
    {
        "thesis_section": "Data Analysis",
        "recommended_asset": "fig01_average_weekly_incidence",
        "reason": "Introduces the temporal behaviour of ARI and ILI."
    },
    {
        "thesis_section": "Data Analysis",
        "recommended_asset": "fig02_mean_incidence_by_country_ari / fig02_mean_incidence_by_country_ili",
        "reason": "Shows cross-country differences in average incidence."
    },
    {
        "thesis_section": "Time Series Analysis",
        "recommended_asset": "fig03_seasonal_profile_by_disease",
        "reason": "Shows seasonal patterns and motivates seasonal/statistical models."
    },
    {
        "thesis_section": "Time Series Analysis",
        "recommended_asset": "fig04_country_correlation_heatmap_ari / fig04_country_correlation_heatmap_ili",
        "reason": "Supports discussion of cross-country similarity and pooled/global models."
    },
    {
        "thesis_section": "Time Series Analysis",
        "recommended_asset": "fig06_acf_representative_series",
        "reason": "Supports the discussion of temporal dependence and lag-based forecasting."
    },
    {
        "thesis_section": "Results",
        "recommended_asset": "07_internal_top5_models_by_scope_disease_horizon",
        "reason": "Main internal comparison table by disease and horizon."
    },
    {
        "thesis_section": "Results",
        "recommended_asset": "08_internal_winners_by_scope_disease_horizon",
        "reason": "Compact table of winning internal methods."
    },
    {
        "thesis_section": "Results",
        "recommended_asset": "fig07_mase_heatmap_*",
        "reason": "Visual summary of model performance by horizon."
    },
    {
        "thesis_section": "Results",
        "recommended_asset": "fig09_forecast_examples_observed_vs_predicted",
        "reason": "Forecast visual examples comparing observed and predicted values."
    },
    {
        "thesis_section": "Results / Diebold-Mariano",
        "recommended_asset": "21_dm_test_simplified_report_table",
        "reason": "Formal statistical comparison of selected procedures."
    },
    {
        "thesis_section": "Results / External Benchmark",
        "recommended_asset": "10_respicast_main_top3_tfg_top2_respicast_baseline",
        "reason": "Final simplified external benchmark using MAE and scaled AE."
    },
    {
        "thesis_section": "Results / External Benchmark",
        "recommended_asset": "fig11_respicast_scaled_AE_ranking_*",
        "reason": "Visual ranking of TFG and RespiCast methods by scaled AE."
    },
    {
        "thesis_section": "Results / External Benchmark",
        "recommended_asset": "fig12_respicast_scaled_AE_ratio_*",
        "reason": "Relative comparison against respicast-quantileBaseline without using pointwise log ratios."
    },
    {
        "thesis_section": "Appendix / External Benchmark Audits",
        "recommended_asset": "14_respicast_zero_error_audit / 15_respicast_prediction_quality_audit",
        "reason": "Documents zero-error cases and quality-control decisions."
    },
])

save_table(
    suggested_order,
    "23_suggested_assets_order_for_memory",
    "Suggested order of figures and tables in the thesis memory.",
    "Appendix / Planning",
    latex=False,
)

display(suggested_order)

print("\nDone.")

PROJECT_ROOT: C:\Users\aolas\UNI PYTHON\TFG
DATA_DIR: C:\Users\aolas\UNI PYTHON\TFG\data
RESULTS_DIR: C:\Users\aolas\UNI PYTHON\TFG\results
OUTPUT: C:\Users\aolas\UNI PYTHON\TFG\results\report_assets
Incidence data loaded using mode: latest_files
truth shape: (17331, 5)
date range: 2014-10-05 00:00:00 to 2024-10-13 00:00:00
diseases: ['ARI', 'ILI']
locations: ['AT', 'BE', 'BG', 'CY', 'CZ', 'DE', 'DK', 'EE', 'ES', 'FI', 'FR', 'GR', 'HR', 'HU', 'IE', 'IS', 'IT', 'LT', 'LU', 'LV', 'MT', 'NL', 'NO', 'PL', 'PT', 'RO', 'SI', 'SK']


,disease,n_countries,first_date,last_date,total_observed_weeks,mean_country_coverage,mean_incidence,median_incidence
0,ARI,19,2014-10-05,2024-10-13,6685,0.671454,1972.668685,1726.392105
1,ILI,26,2014-10-05,2024-10-13,10646,0.781415,583.096048,351.842308


EDA figures restricted to final modelling subsets:
ARI countries: ['BE', 'BG', 'CZ', 'DE', 'EE', 'LT', 'RO', 'SI'] | expected: ['BE', 'BG', 'CZ', 'DE', 'EE', 'LT', 'RO', 'SI'] | rows: 4136 | max value: 7170.1
ILI countries: ['BE', 'CZ', 'EE', 'GR', 'IE', 'LT', 'NL', 'PL', 'RO', 'SI'] | expected: ['BE', 'CZ', 'EE', 'GR', 'IE', 'LT', 'NL', 'PL', 'RO', 'SI'] | rows: 5189 | max value: 2302.5


,disease,location,truth_date,value,source_file
3994,ARI,SI,2022-01-30,7170.1,latest-ARI_incidence.csv
3995,ARI,SI,2022-02-06,6583.3,latest-ARI_incidence.csv
3993,ARI,SI,2022-01-23,4476.9,latest-ARI_incidence.csv
3996,ARI,SI,2022-02-13,3387.7,latest-ARI_incidence.csv
3890,ARI,SI,2020-02-02,3335.2,latest-ARI_incidence.csv
4041,ARI,SI,2022-12-25,3299.9,latest-ARI_incidence.csv
1989,ARI,DE,2022-12-18,3251.0,latest-ARI_incidence.csv
3992,ARI,SI,2022-01-16,3233.1,latest-ARI_incidence.csv
1988,ARI,DE,2022-12-11,3211.0,latest-ARI_incidence.csv
1738,ARI,DE,2018-02-25,2946.8,latest-ARI_incidence.csv


Internal result files used:
- C:\Users\aolas\UNI PYTHON\TFG\results\final_test_2024_2025\finaltest_horizon_summary.csv
- C:\Users\aolas\UNI PYTHON\TFG\results\final_test_2024_2025\finaltest_common6_horizon_summary.csv

Internal global result coverage:


...

,scope,disease,h,n_rows,n_tfg,n_respicast,n_pairs_min,n_pairs_max,reference_pairs_min,reference_pairs_max
0,common6,ARI,1,6,3,3,232,358,358,358
1,common6,ARI,2,6,3,3,228,352,352,352
2,common6,ARI,3,6,3,3,279,346,346,346
3,common6,ARI,4,6,3,3,273,340,340,340
4,common6,ILI,1,6,3,3,317,358,358,358
5,common6,ILI,2,6,3,3,311,352,352,352
6,common6,ILI,3,6,3,3,306,346,346,346
7,common6,ILI,4,6,3,3,300,340,340,340
8,global,ARI,1,6,3,3,322,480,480,480
9,global,ARI,2,6,3,3,317,472,472,472



REPORT ASSETS GENERATED
Output folder: C:\Users\aolas\UNI PYTHON\TFG\results\report_assets
Catalog: C:\Users\aolas\UNI PYTHON\TFG\results\report_assets\report_assets_catalog.csv

Figures: 58
Tables : 49


,type,name,path,description,section_hint
0,table,00_incidence_file_inventory,results\report_assets\tables\00_incidence_file...,Input incidence files used for report EDA.,Data Collection
1,table,01_data_coverage_by_country_disease,results\report_assets\tables\01_data_coverage_...,Coverage and descriptive statistics by country...,Data Collection / Data Analysis
2,latex_table,01_data_coverage_by_country_disease,results\report_assets\tables\01_data_coverage_...,Coverage and descriptive statistics by country...,Data Collection / Data Analysis
3,table,02_data_coverage_by_disease,results\report_assets\tables\02_data_coverage_...,Overall data coverage by disease.,Data Collection / Data Analysis
4,latex_table,02_data_coverage_by_disease,results\report_assets\tables\02_data_coverage_...,Overall data coverage by disease.,Data Collection / Data Analysis
...,...,...,...,...,...
102,figure,fig13_respicast_coverage_vs_baseline,results\report_assets\figures\fig13_respicast_...,Average coverage relative to respicast-quantil...,Results / External Benchmark
103,figure,fig13_respicast_coverage_vs_baseline,results\report_assets\figures\fig13_respicast_...,Average coverage relative to respicast-quantil...,Results / External Benchmark
104,table,29_plot_data_fig13_respicast_coverage_vs_baseline,results\report_assets\tables\29_plot_data_fig1...,Exact data used for fig13 RespiCast coverage p...,Results / External Benchmark / Plot Data
105,table,20_dm_file_inventory,results\report_assets\tables\20_dm_file_invent...,DM-test files detected in results/.,Results / Diebold-Mariano


,thesis_section,recommended_asset,reason
0,Data Collection / Data Preprocessing,01_data_coverage_by_country_disease,Documents which countries and diseases are ava...
1,Data Analysis,fig01_average_weekly_incidence,Introduces the temporal behaviour of ARI and ILI.
2,Data Analysis,fig02_mean_incidence_by_country_ari / fig02_me...,Shows cross-country differences in average inc...
3,Time Series Analysis,fig03_seasonal_profile_by_disease,Shows seasonal patterns and motivates seasonal...
4,Time Series Analysis,fig04_country_correlation_heatmap_ari / fig04_...,Supports discussion of cross-country similarit...
5,Time Series Analysis,fig06_acf_representative_series,Supports the discussion of temporal dependence...
6,Results,07_internal_top5_models_by_scope_disease_horizon,Main internal comparison table by disease and ...
7,Results,08_internal_winners_by_scope_disease_horizon,Compact table of winning internal methods.
8,Results,fig07_mase_heatmap_*,Visual summary of model performance by horizon.
9,Results,fig09_forecast_examples_observed_vs_predicted,Forecast visual examples comparing observed an...



Done.
